[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/autoencoder_vae.ipynb)

# Variational Autoencoders (VAE)

In a VAE, the encoder doesn't produce a single point of the latent space, but a probability distribution: a Gaussian with mean $\mu$ and variance $\sigma^2$. The decoder reconstructs the image from a point $z$ sampled from that distribution.

The loss has two terms:

* the **reconstruction** error (binary cross-entropy between the image and its reconstruction);
* the **KL divergence** between the distribution of each image and a standard normal $N(0, I)$, which keeps the latent space continuous and centred at 0.

Because of this, we can **generate new images** by decoding points of the latent space.

Import the libraries.

In [ ]:
import torch
from torchvision import datasets
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from torchvision.utils import save_image
from os.path import exists
import torch.nn.functional as F

Utility functions, to store the temporary images and to convert matrix shape.

In [ ]:
os.makedirs('./vae_img', exist_ok=True)


def to_img(x):
    x = x.clamp(0, 1)
    x = x.view(x.size(0), 1, 28, 28)
    return x

## Dataset

In [ ]:
num_epochs = 20
batch_size = 128
learning_rate = 1e-3

# Transforms images to a PyTorch Tensor, with values in [0, 1]
# (needed by the binary cross-entropy; the decoder ends with a sigmoid)
img_transform = transforms.ToTensor()

# Download the MNIST Dataset
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=img_transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=img_transform)

# DataLoader is used to load the dataset
# for training
loader = torch.utils.data.DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)

## Model

In [ ]:
class VAE(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = torch.nn.Linear(784, 400)
        self.fc21 = torch.nn.Linear(400, 2)  # mean
        self.fc22 = torch.nn.Linear(400, 2)  # log variance
        self.fc3 = torch.nn.Linear(2, 400)
        self.fc4 = torch.nn.Linear(400, 784)

    def encode(self, x):
        h1 = F.relu(self.fc1(x))
        return self.fc21(h1), self.fc22(h1)

    def reparametrize(self, mu, logvar):
        # z = mu + sigma * eps, with eps ~ N(0, 1)
        # (the reparametrization trick: the randomness is in eps, so the gradients flow through mu and sigma)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h3 = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(h3))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
# Model Initialization
model = VAE()
#moving to gpu
model = model.to(device)


def loss_function(recon_x, x, mu, logvar):
    """
    recon_x: generating images
    x: origin images
    mu: latent mean
    logvar: latent log variance
    """
    # reconstruction error
    BCE = F.binary_cross_entropy(recon_x, x, reduction="sum")
    # KL divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD


optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## Training / Loading

If the model was already trained and saved, it is loaded from the file. Otherwise, it is trained (delete the file to train it again).

In [ ]:
model_file = "autoencoder_vae.pth"

In [ ]:
if exists(model_file):
    model.load_state_dict(torch.load(model_file, map_location=device, weights_only=True))
    model.eval()
else:
    losses = []
    model.train()
    for epoch in range(num_epochs):
        train_loss = 0
        for batch_idx, data in enumerate(loader):
            img, _ = data
            img = img.view(img.size(0), -1)
            img = img.to(device)
            optimizer.zero_grad()
            recon_batch, mu, logvar = model(img)
            loss = loss_function(recon_batch, img, mu, logvar)
            loss.backward()
            train_loss += loss.item()
            optimizer.step()

        # average loss per image
        losses.append(train_loss / len(loader.dataset))
        print('====> Epoch: {} Average loss: {:.4f}'.format(epoch + 1, losses[-1]))
        if epoch % 5 == 0:
            save = to_img(recon_batch.cpu().data)
            save_image(save, './vae_img/image_{}.png'.format(epoch))

    torch.save(model.state_dict(), model_file)
    model.eval()

    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.plot(losses)
    plt.show()

## Display sample images

Test images (top) and their reconstructions (bottom).

In [ ]:
import random
from mpl_toolkits.axes_grid1 import ImageGrid


def show_reconstructions(dataset, filename=None):
    """Shows 10 random images (top row) and their reconstructions (bottom row)."""
    model.eval()
    fig = plt.figure(figsize=(20., 5.))
    grid = ImageGrid(fig, 111,  # similar to subplot(111)
                     nrows_ncols=(2, 10),  # creates 2x10 grid of axes
                     axes_pad=0.1,  # pad between axes in inch.
                     )

    for i in range(10):
        image, _ = dataset[random.randrange(len(dataset))]

        img = image.reshape(-1, 28*28).to(device)
        with torch.no_grad():
            reconstructed, _, _ = model(img)
        item = reconstructed.reshape(-1, 28, 28).cpu().numpy()

        # Iterating over the grid returns the Axes.
        grid[i].imshow(image[0], cmap='gray')
        grid[i].axis('off')
        grid[10+i].imshow(item[0], cmap='gray')
        grid[10+i].axis('off')

    if filename:
        plt.savefig(filename, transparent=True)
    plt.show()


show_reconstructions(test_dataset, "nums_vae.png")

## Display latent space

Each test image is represented by the mean $\mu$ of its distribution in the latent space. Compare with the latent space of the autoencoder: the KL divergence keeps the points around $(0, 0)$, without large gaps.

In [ ]:
import seaborn as sns
import pandas as pd

# encode all the test images (in batches)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000)
latents = []
labels = []
model.eval()
with torch.no_grad():
    for images, label in test_loader:
        images = images.view(images.size(0), -1).to(device)
        mu, logvar = model.encode(images)
        latent = mu  # the mean of the distribution of each image
        latents.append(latent.cpu().numpy())
        labels.append(label.numpy())

df = pd.DataFrame(np.concatenate(latents), columns=("x1", "x2"))
df["label"] = np.concatenate(labels)

f, ax = plt.subplots(figsize=(12, 12))
sns.scatterplot(x="x1", y="x2",
                hue="label",
                legend="full",
                palette="tab10",
                linewidth=0,
                s=5,
                data=df, ax=ax)
plt.savefig("latent_space.png", transparent=True)
plt.show()

## Generate new digits

Since the latent space is continuous and centred at 0, we can decode points that don't correspond to any image of the dataset. Here, we decode a grid of points $(x_1, x_2)$ between -3 and 3. Compare the digits with the latent space above: they change smoothly along the grid.

In [ ]:
n = 15
values = np.linspace(-3, 3, n)
model.eval()
with torch.no_grad():
    # one point for each pair (x1, x2): x1 changes along the columns and x2 along the rows (from top to bottom: 3 to -3)
    z = torch.tensor([[x1, x2] for x2 in values[::-1] for x1 in values], dtype=torch.float32).to(device)
    generated = model.decode(z).cpu().reshape(n, n, 28, 28)

# put the images side by side in a single image
canvas = generated.permute(0, 2, 1, 3).reshape(n * 28, n * 28).numpy()

plt.figure(figsize=(10, 10))
plt.imshow(canvas, cmap='gray', extent=[-3, 3, -3, 3])
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Images generated from the latent space")
plt.show()

## Testing on different space

How does the VAE, trained with digits, reconstruct images of another kind? Let's start by visualizing again some MNIST samples.

Visualize some MNIST samples

In [ ]:
show_reconstructions(test_dataset)

### Exercise

Download the Fashion-MNIST test set with torchvision (`datasets.FashionMNIST`, with the same transform), and visualize how the VAE reconstructs its images. What happens? Why?

In [ ]:
# Download the Fashion MNIST Dataset
fashion_dataset = ...

Visualize some Fashion MNIST samples

In [ ]:
...